In [1]:
import pandas as pd
import numpy as np

In [2]:
#LOAD RAW DATA

df= pd.read_csv(r"C:\Users\nivet\Downloads\retail-orders-raw.csv")
print("File has been uploaded Successfully")

File has been uploaded Successfully


In [3]:
#=========================================================
# DATA UNDERSTANDING
#=========================================================

In [4]:
print("Rows :", len(df))
print("Columns :", len(df.columns))

Rows : 12
Columns : 9


In [5]:
df.head()

,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799,10.0,Paid
1,RT-1002,03/01/2026,Fresher,Bengaluru,Course Access,1,1499,0.0,paid
2,RT-1003,2026-01-05,student,Chennai,Course Access,1,1499,NaN,Pending
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
4,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          12 non-null     object 
 1   order_date        11 non-null     object 
 2   customer_segment  12 non-null     object 
 3   city              11 non-null     object 
 4   category          12 non-null     object 
 5   quantity          12 non-null     object 
 6   unit_price        12 non-null     int64  
 7   discount_pct      11 non-null     float64
 8   payment_status    12 non-null     object 
dtypes: float64(1), int64(1), object(7)
memory usage: 996.0+ bytes


In [7]:
#========================================================
#DATA PROFILING 
#========================================================

In [8]:
missing = pd.DataFrame({
    "Columns": df.columns,
    "missing_count" : df.isna().sum(),
    "missing_percentage" : (df.isna().mean()*100).round(2)
})

missing

,Columns,missing_count,missing_percentage
order_id,order_id,0,0.00
order_date,order_date,1,8.33
customer_segment,customer_segment,0,0.00
city,city,1,8.33
category,category,0,0.00
quantity,quantity,0,0.00
unit_price,unit_price,0,0.00
discount_pct,discount_pct,1,8.33
payment_status,payment_status,0,0.00


In [9]:
#duplicate_orders

duplicate_orders = df[df["order_id"].duplicated(keep=False)]
print("Duplicate rows: " ,len(duplicate_orders))
print("Duplicate Order IDs:")
print(duplicate_orders["order_id"].unique())

Duplicate rows:  2
Duplicate Order IDs:
['RT-1004']


In [10]:

quantity_numeric= pd.to_numeric( df["quantity"], errors="coerce")
invalid_quantity = df[quantity_numeric.isna() | (quantity_numeric <=0)]

print("Invalid Quantity records :", len(invalid_quantity))
invalid_quantity[["order_id", "quantity"]]

Invalid Quantity records : 2


,order_id,quantity
6,RT-1006,-1
8,RT-1008,two


In [11]:
# validity of unit price

invalid_price= df[pd.to_numeric(df["unit_price"], errors="coerce").isna() |
    (pd.to_numeric(df["unit_price"],errors="coerce") < 0)
    ]

print("Invalid Unit Price :", len(invalid_price))
invalid_price[["order_id", "unit_price"]]

Invalid Unit Price : 0


,order_id,unit_price


In [12]:
#validity of Discount

discount_numeric = pd.to_numeric(
    df["discount_pct"], errors= "coerce")

invalid_discount= df[ discount_numeric.notna() &
    ((discount_numeric <0) | (discount_numeric > 100))
    ]

print("Invalid discounts: ", len(invalid_discount))
invalid_discount[["order_id", "discount_pct"]]

Invalid discounts:  1


,order_id,discount_pct
7,RT-1007,105.0


In [13]:
#validity of order date

changed_dates = pd.to_datetime(
    df["order_date"], errors="coerce")

invalid_date = df[df["order_date"].notna() & changed_dates.isna() ]

print("Invalid / Unchanged Dates:", len(invalid_date))
invalid_date= [["order_id", "order_date"]]

Invalid / Unchanged Dates: 2


In [14]:
#customer consistency check

print(df["customer_segment"].value_counts(dropna=False))

customer_segment
Student         4
Professional    4
Fresher         3
student         1
Name: count, dtype: int64


In [15]:
# checking the payment status

print(df["payment_status"].value_counts(dropna=False))

payment_status
Paid        7
Pending     2
paid        1
Failed      1
Refunded    1
Name: count, dtype: int64


In [16]:
#Freshness Check (latest )

valid_dates = pd.to_datetime(
    df["order_date"], errors= "coerce")

latest_date= valid_dates.max()
earliest_date=valid_dates.min()

print("Earliest valid order date: ", earliest_date)
print("Latest valid order date: ", latest_date)

Earliest valid order date:  2026-01-03 00:00:00
Latest valid order date:  2026-01-18 00:00:00


In [17]:
#Profiling Summary

profile = pd.DataFrame({
    "Dimension" : [
        "Completeness",
        "Uniqueness",
        "Quantity Validity", 
        "Unit Price Validity",
        "Discount Validity",
        "Date Validity",
        "Category consistency",
        "Payment consistency",
        "Freshness" ],
    "Status" :[
        "CHECKED", "CHECKED", "CHECKED","CHECKED","CHECKED",
        "CHECKED","CHECKED","CHECKED","CHECKED"]
})

profile

,Dimension,Status
0,Completeness,CHECKED
1,Uniqueness,CHECKED
2,Quantity Validity,CHECKED
3,Unit Price Validity,CHECKED
4,Discount Validity,CHECKED
5,Date Validity,CHECKED
6,Category consistency,CHECKED
7,Payment consistency,CHECKED
8,Freshness,CHECKED


In [18]:
#==============================================================
# DATA CLEANING & VALIDATION
#==============================================================

In [19]:
clean_df=df.copy()

In [20]:
clean_df = clean_df.drop_duplicates(
    subset= "order_id", keep= "first"
)

In [21]:
print("Rows before: ", len(df))
print("Rows after : ", len(clean_df))

Rows before:  12
Rows after :  11


In [22]:
clean_df["order_id"].is_unique

True

In [23]:
# Parse YYYY-MM-DD dates
clean_df["order_date_parsed"] = pd.to_datetime(
    clean_df["order_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

# Parse remaining DD/MM/YYYY dates
mask = (
    clean_df["order_date_parsed"].isna() &
    clean_df["order_date"].notna()
)

clean_df.loc[mask, "order_date_parsed"] = pd.to_datetime(
    clean_df.loc[mask, "order_date"],
    format="%d/%m/%Y",
    errors="coerce"
)

# Convert valid dates to ISO format
clean_df["order_date"] = (
    clean_df["order_date_parsed"]
    .dt.strftime("%Y-%m-%d")
)

# Show dates that could not be converted
clean_df[
    clean_df["order_date_parsed"].isna()
][["order_id", "order_date"]]

,order_id,order_date
6,RT-1006,NaN
11,RT-1011,NaN


In [24]:
clean_df["customer_segment"] = (
    clean_df["customer_segment"]
    .str.strip()
    .str.title()
)

In [25]:
clean_df["customer_segment"].value_counts()

customer_segment
Student         5
Fresher         3
Professional    3
Name: count, dtype: int64

In [26]:
missing_city = (
    clean_df["city"].isna() |
    (clean_df["city"].str.strip() == "")
)

clean_df.loc[missing_city, ["order_id", "city"]]

,order_id,city
5,RT-1005,NaN


In [27]:
allowed_categories = [
    "Learning Kit",
    "Course Access",
    "Mentor Session"
]

invalid_category = clean_df[
    ~clean_df["category"].isin(allowed_categories)
]

invalid_category[["order_id", "category"]]

,order_id,category


In [28]:
clean_df["quantity"] = (
    clean_df["quantity"]
    .replace({"two": 2})
)

clean_df["quantity"] = pd.to_numeric(
    clean_df["quantity"],
    errors="coerce"
)

In [29]:
invalid_quantity = clean_df[
    clean_df["quantity"].isna() |
    (clean_df["quantity"] <= 0)
]

invalid_quantity[["order_id", "quantity"]]

,order_id,quantity
6,RT-1006,-1


In [30]:
clean_df["unit_price"] = pd.to_numeric(
    clean_df["unit_price"],
    errors="coerce"
)

invalid_price = clean_df[
    clean_df["unit_price"].isna() |
    (clean_df["unit_price"] < 0)
]

invalid_price[["order_id", "unit_price"]]

,order_id,unit_price


In [31]:
clean_df["discount_pct"] = pd.to_numeric(
    clean_df["discount_pct"],
    errors="coerce"
)

invalid_discount = clean_df[
    clean_df["discount_pct"].notna() &
    (
        (clean_df["discount_pct"] < 0) |
        (clean_df["discount_pct"] > 100)
    )
]

invalid_discount[["order_id", "discount_pct"]]

,order_id,discount_pct
7,RT-1007,105.0


In [32]:
clean_df["discount_pct"] = clean_df["discount_pct"].fillna(0)

In [33]:
clean_df["payment_status"] = (
    clean_df["payment_status"]
    .str.strip()
    .str.title()
)

clean_df["payment_status"].value_counts()

payment_status
Paid        7
Pending     2
Failed      1
Refunded    1
Name: count, dtype: int64

In [34]:
reject_ids = [
    "RT-1005",
    "RT-1006",
    "RT-1007",
    "RT-1011"
]

rejected_df = clean_df[
    clean_df["order_id"].isin(reject_ids)
].copy()

approved_df = clean_df[
    ~clean_df["order_id"].isin(reject_ids)
].copy()

print("Approved records:", len(approved_df))
print("Rejected records:", len(rejected_df))

Approved records: 7
Rejected records: 4


In [35]:
approved_df = approved_df.drop(
    columns=["order_date_parsed"]
)

rejected_df = rejected_df.drop(
    columns=["order_date_parsed"]
)

In [36]:
approved_df.to_csv(
    "retail-orders-cleaned.csv",
    index=False
)

rejected_df.to_csv(
    "retail-orders-rejected.csv",
    index=False
)